In [ ]:
# 快速了解原始数据
import pandas as pd

# 加载原始训练数据
df_raw = pd.read_parquet('../data/raw/us_census_20260215_201556.parquet')

print("📊 原始数据快速浏览\n")
print(f"数据形状: {df_raw.shape[0]:,}行 × {df_raw.shape[1]}列")
print(f"时间范围: {df_raw['date'].min().date()} 至 {df_raw['date'].max().date()}")
print(f"总月数: {df_raw['date'].nunique()}个月")
print(f"供应国数: {df_raw['country'].nunique()}个")
print(f"HS代码数: {df_raw['hs_code'].nunique()}个")

print("\n" + "="*80)
print("前5条记录示例:")
print("="*80)
display(df_raw.head())

print("\n" + "="*80)
print("四类芯片产品贸易额分布:")
print("="*80)
hs_summary = df_raw.groupby('hs_code').agg({
    'value_usd': ['sum', 'count'],
    'country': 'nunique'
})
hs_summary.columns = ['总贸易额(USD)', '记录数', '供应国数']
hs_summary['总贸易额(十亿)'] = hs_summary['总贸易额(USD)'] / 1e9
hs_summary['占比%'] = hs_summary['总贸易额(USD)'] / hs_summary['总贸易额(USD)'].sum() * 100

# 添加产品名称
hs_names = {
    '854231': 'Processors (处理器)',
    '854232': 'Memories (存储器)',
    '854233': 'Amplifiers (放大器)',
    '854239': 'Other ICs (其他芯片)'
}
hs_summary['产品名称'] = [hs_names.get(idx, idx) for idx in hs_summary.index]

display(hs_summary[['产品名称', '总贸易额(十亿)', '占比%', '记录数', '供应国数']].round(2))

print("\n" + "="*80)
print("Top 10 供应国/地区:")
print("="*80)
top_suppliers = df_raw.groupby('country')['value_usd'].sum().sort_values(ascending=False).head(10)
top_df = pd.DataFrame({
    '国家/地区': top_suppliers.index,
    '总贸易额(十亿美元)': (top_suppliers.values / 1e9).round(2),
    '占比%': (top_suppliers.values / df_raw['value_usd'].sum() * 100).round(1)
})
top_df.index = range(1, 11)
display(top_df)

print("\n💡 关键观察:")
print("  • 处理器(854231)占总贸易额60%以上，是最核心的产品")
print("  • Top 10供应商/地区包含多个区域组织（APEC、ASEAN等）")
print("  • 实际国家级供应商主要是马来西亚、台湾等")
print("  • 10年间总进口额达$1.69万亿，显示美国对进口芯片的高度依赖")

---

## 📚 数据说明：我们分析的是什么？

### 数据来源
- **数据源**: 美国人口普查局 (US Census Bureau)
- **数据类型**: 美国进口贸易数据
- **时间范围**: 2010年1月 - 2019年12月 (10年，120个月)
- **产品类别**: 半导体集成电路 (Integrated Circuits)

### 为什么研究半导体供应链？

半导体是现代电子产品的核心，从手机、电脑到汽车、服务器都离不开芯片。美国高度依赖进口半导体，供应链风险直接影响：
- 🏭 **制造业**: 汽车、电子产品生产
- 💻 **科技行业**: 数据中心、AI计算
- 🛡️ **国家安全**: 军事装备、关键基础设施
- 📱 **消费电子**: 智能手机、电脑供应

### 数据结构

每条记录代表：**一个月 × 一个国家 × 一种芯片类型**

**示例**: 2010年1月，美国从马来西亚进口5445万美元的处理器芯片

| 字段 | 含义 | 示例 |
|------|------|------|
| `date` | 月度日期 | 2010-01-01 |
| `hs_code` | 产品编码 | 854231 |
| `country` | 供应国 | Malaysia (马来西亚) |
| `value_usd` | 进口金额 | $54,453,523 |
| `quantity` | 进口数量 | 0 (部分数据未报告) |
| `trade_type` | 贸易类型 | imports (进口) |

### 四类芯片产品 (HS代码)

| HS代码 | 产品类型 | 中文名称 | 用途 | 贸易额 |
|--------|----------|----------|------|--------|
| **854231** | Processors & Controllers | 处理器和控制器 | CPU、GPU、MCU | $1,025B (60.6%) |
| **854232** | Memories | 存储器 | RAM、闪存 | $174B (10.3%) |
| **854233** | Amplifiers | 放大器 | 信号放大、功率管理 | $49B (2.9%) |
| **854239** | Other Electronic ICs | 其他集成电路 | 各类专用芯片 | $444B (26.2%) |

### 数据规模

- **总记录数**: 37,756条
- **总贸易额**: $1.69万亿美元 (十年)
- **月均贸易额**: $141亿美元
- **供应国数**: 214个国家/地区
- **主要供应商**: APEC、亚洲、太平洋沿岸国家、马来西亚

### 供应链风险类型

我们通过数据分析识别以下风险：

1. **集中度风险** (Concentration Risk)
   - 少数国家控制大部分供应
   - 指标: HHI, Top-N份额
   - 阈值: HHI > 2500 或 Top1 > 50%

2. **波动性风险** (Volatility Risk)
   - 贸易额大幅波动
   - 指标: CoV (变异系数)
   - 阈值: CoV > 30%

3. **增长风险** (Growth Risk)
   - 贸易额持续下降
   - 指标: YoY增长率
   - 阈值: YoY < -10%

4. **依赖风险** (Dependency Risk)
   - 供应商数量少
   - 指标: 供应商数量
   - 阈值: 少于5个主要供应商

---

### 模型训练建议

基于以上分析，对模型开发提出以下建议：

#### 1. 特征选择优先级

**高优先级特征** (直接反映风险):
- `hhi` - 供应链集中度核心指标
- `value_cov` - 波动性核心指标
- `stability_score` - 综合稳定性评估
- `top1_share` - 单一供应商依赖度
- `growth_yoy` - 同比增长趋势

**中优先级特征** (补充信息):
- 滚动窗口特征 (`hhi_3m`, `value_cov_6m` 等)
- 移动平均特征 (`value_ma_*`)
- 季节性特征 (`seasonality_index`, `is_peak_season`)

**低优先级特征** (可能冗余):
- 高度相关的特征对 (|r| > 0.9)
- 滞后特征 (如果使用时间序列模型)

#### 2. 特征工程建议

- **交互特征**: 考虑 `hhi * value_cov` (集中度 × 波动性)
- **风险评分**: 创建综合风险指标作为监督学习目标
- **差分特征**: `hhi_change_mom`, `growth_acceleration` 捕捉变化趋势
- **时间衰减**: 对历史数据应用指数衰减权重

#### 3. 模型架构建议

**时间序列模型** (预测未来风险):
- LSTM/GRU for 序列模式学习
- Prophet for 季节性和趋势分解
- ARIMA for 短期预测

**分类模型** (风险等级分类):
- XGBoost/LightGBM for 表格数据
- Random Forest for 特征重要性分析
- Logistic Regression for 可解释性

**异常检测模型** (识别异常模式):
- Isolation Forest
- Autoencoder
- One-Class SVM

#### 4. 数据分割策略

- **时间分割**: 2010-2016训练, 2017-2018验证, 2019测试
- **HS代码分组**: 确保每组包含所有HS代码
- **交叉验证**: 使用TimeSeriesSplit保持时间顺序

#### 5. 评估指标

- **分类任务**: Precision, Recall, F1 (风险识别)
- **回归任务**: MAE, RMSE, MAPE (风险评分预测)
- **排序任务**: NDCG, MAP (风险排序)

---

**下一步**: 提取完整训练数据特征 (37,756条记录) 并开始模型训练

In [ ]:
# 关键发现总结
print("=" * 80)
print("关键发现总结")
print("=" * 80)

print("\n1. 供应链集中度:")
print(f"   - 平均HHI: {df['hhi'].mean():.0f} (中等集中度)")
print(f"   - 高度集中占比: {(df['hhi'] > 2500).sum() / len(df) * 100:.1f}%")
print(f"   - 最大供应商平均份额: {df['top1_share'].mean() * 100:.1f}%")

print("\n2. 供应链稳定性:")
print(f"   - 平均波动性 (CoV): {df['value_cov'].mean() * 100:.1f}%")
print(f"   - 平均稳定性评分: {df['stability_score'].mean():.1f}/100")
print(f"   - 高波动占比: {(df['value_cov'] > 0.3).sum() / len(df) * 100:.1f}%")

if 'growth_mom' in df.columns:
    print("\n3. 增长趋势:")
    print(f"   - 平均MoM增长: {df['growth_mom'].mean() * 100:.1f}%")
    print(f"   - 负增长占比: {(df['growth_mom'] < 0).sum() / len(df) * 100:.1f}%")

print("\n4. 贸易规模:")
total_trade = df['value_usd'].sum()
print(f"   - 总贸易额: ${total_trade / 1e9:.2f}B")
print(f"   - 月均贸易额: ${df.groupby('date')['value_usd'].sum().mean() / 1e6:.2f}M")

print("\n5. 供应商多样性:")
print(f"   - 总供应国数: {df['country'].nunique()}")
print(f"   - Top 5国家份额: {df.groupby('country')['value_usd'].sum().nlargest(5).sum() / total_trade * 100:.1f}%")

print("\n" + "=" * 80)

## 9. 总结与建议

In [ ]:
# 高风险时期识别
if 'composite_risk_score' in df_risk.columns:
    high_risk_threshold = 70
    high_risk_periods = df_risk[df_risk['composite_risk_score'] > high_risk_threshold]
    
    if len(high_risk_periods) > 0:
        print(f"\n高风险时期 (风险评分 > {high_risk_threshold}):\n")
        high_risk_summary = high_risk_periods.groupby(['date', 'hs_code']).agg({
            'composite_risk_score': 'mean',
            'hhi': 'mean',
            'value_cov': 'mean',
            'country': 'count'
        }).round(2)
        high_risk_summary.columns = ['风险评分', 'HHI', 'CoV', '记录数']
        print(high_risk_summary.head(10))
    else:
        print(f"\n✅ 无高风险时期 (所有评分 < {high_risk_threshold})")

In [ ]:
# 综合风险评分分布
if 'composite_risk_score' in df_risk.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # 直方图
    axes[0].hist(df_risk['composite_risk_score'].dropna(), bins=30, 
                 edgecolor='black', alpha=0.7, color='crimson')
    axes[0].axvline(50, color='orange', linestyle='--', label='中等风险 (50)')
    axes[0].axvline(70, color='red', linestyle='--', label='高风险 (70)')
    axes[0].set_xlabel('综合风险评分')
    axes[0].set_ylabel('频数')
    axes[0].set_title('综合风险评分分布')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # 时间序列
    risk_ts = df_risk.groupby('date')['composite_risk_score'].mean()
    axes[1].plot(risk_ts.index, risk_ts.values, marker='o', linewidth=2, color='darkred')
    axes[1].axhline(50, color='orange', linestyle='--', alpha=0.7)
    axes[1].axhline(70, color='red', linestyle='--', alpha=0.7)
    axes[1].set_xlabel('日期')
    axes[1].set_ylabel('平均风险评分')
    axes[1].set_title('综合风险评分时间演变')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 风险标志定义
df_risk = df.copy()

# 集中度风险
df_risk['risk_high_concentration'] = (df_risk['hhi'] > 2500) | (df_risk['top1_share'] > 0.5)

# 波动性风险
df_risk['risk_high_volatility'] = (df_risk['value_cov'] > 0.3) | (df_risk['stability_score'] < 60)

# 增长风险
if 'growth_yoy' in df_risk.columns:
    df_risk['risk_negative_growth'] = df_risk['growth_yoy'] < -0.1  # YoY下降超过10%

# 综合风险评分 (0-100, 越高风险越大)
risk_components = []
if 'hhi' in df_risk.columns:
    risk_components.append((df_risk['hhi'] / 100).clip(0, 100))
if 'value_cov' in df_risk.columns:
    risk_components.append((df_risk['value_cov'] * 100).clip(0, 100))
if 'stability_score' in df_risk.columns:
    risk_components.append(100 - df_risk['stability_score'])

if risk_components:
    df_risk['composite_risk_score'] = sum(risk_components) / len(risk_components)

# 风险统计
print("风险指标统计:\n")
print(f"高集中度风险: {df_risk['risk_high_concentration'].sum()} 记录 ({df_risk['risk_high_concentration'].sum()/len(df_risk)*100:.1f}%)")
print(f"高波动性风险: {df_risk['risk_high_volatility'].sum()} 记录 ({df_risk['risk_high_volatility'].sum()/len(df_risk)*100:.1f}%)")
if 'risk_negative_growth' in df_risk.columns:
    print(f"负增长风险: {df_risk['risk_negative_growth'].sum()} 记录 ({df_risk['risk_negative_growth'].sum()/len(df_risk)*100:.1f}%)")

if 'composite_risk_score' in df_risk.columns:
    print(f"\n综合风险评分:")
    print(f"  平均值: {df_risk['composite_risk_score'].mean():.2f}")
    print(f"  中位数: {df_risk['composite_risk_score'].median():.2f}")
    print(f"  高风险 (>70): {(df_risk['composite_risk_score'] > 70).sum()} ({(df_risk['composite_risk_score'] > 70).sum()/len(df_risk)*100:.1f}%)")

## 8. 风险识别与预警

In [ ]:
# 各HS代码的Top供应国
for hs in df['hs_code'].unique()[:4]:  # 前4个HS代码
    df_hs = df[df['hs_code'] == hs]
    top_countries_hs = df_hs.groupby('country')['value_usd'].sum().sort_values(ascending=False).head(5)
    
    print(f"\nHS {hs} - Top 5供应国:")
    for i, (country, value) in enumerate(top_countries_hs.items(), 1):
        pct = value / df_hs['value_usd'].sum() * 100
        print(f"  {i}. {country}: ${value/1e6:.2f}M ({pct:.1f}%)")

In [ ]:
# Top供应国分析
top_countries = df.groupby('country')['value_usd'].sum().sort_values(ascending=False).head(15)
top_countries_pct = (top_countries / top_countries.sum() * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 柱状图
top_countries_million = top_countries / 1e6
top_countries_million.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_xlabel('贸易额 (百万美元)')
axes[0].set_ylabel('国家')
axes[0].set_title('Top 15 供应国贸易额')
axes[0].grid(True, alpha=0.3, axis='x')

# 饼图
top_countries_pct.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_ylabel('')
axes[1].set_title('Top 15 供应国份额分布')

plt.tight_layout()
plt.show()

print("Top 5 供应国:")
for i, (country, value) in enumerate(top_countries.head(5).items(), 1):
    pct = value / df['value_usd'].sum() * 100
    print(f"  {i}. {country}: ${value/1e6:.2f}M ({pct:.1f}%)")

## 7. 国家级供应链分析

In [ ]:
# 完整特征相关性矩阵 (仅显示数值型特征)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# 移除原始数据列
exclude_cols = ['value_usd', 'quantity', 'month', 'quarter', 'year']
feature_only_cols = [c for c in numeric_cols if c not in exclude_cols]

if len(feature_only_cols) > 3:
    corr_all = df[feature_only_cols].corr()
    
    plt.figure(figsize=(20, 18))
    sns.heatmap(corr_all, cmap='coolwarm', center=0, 
                square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
                annot=False)  # 太多特征不显示数值
    plt.title('所有特征相关性矩阵')
    plt.tight_layout()
    plt.show()
    
    print(f"✅ 相关性矩阵已生成 ({len(feature_only_cols)} x {len(feature_only_cols)})")

In [ ]:
# 核心风险指标相关性热图
risk_features = ['hhi', 'top1_share', 'value_cov', 'stability_score', 'growth_mom', 'growth_yoy']
available_risk_features = [f for f in risk_features if f in df.columns]

if len(available_risk_features) > 1:
    corr = df[available_risk_features].corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
                square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('核心风险指标相关性热图')
    plt.tight_layout()
    plt.show()
    
    # 找出强相关特征对
    print("强相关特征对 (|r| > 0.7):")
    for i in range(len(corr.columns)):
        for j in range(i+1, len(corr.columns)):
            if abs(corr.iloc[i, j]) > 0.7:
                print(f"  {corr.columns[i]} <-> {corr.columns[j]}: {corr.iloc[i, j]:.3f}")

## 6. 特征相关性分析

In [ ]:
# 增长率时间序列
if 'growth_mom' in df.columns:
    df_growth = df.groupby('date')['growth_mom'].mean() * 100
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_growth.index, y=df_growth.values, 
                             mode='lines+markers', name='MoM增长率',
                             line=dict(color='blue')))
    fig.add_hline(y=0, line_dash="solid", line_color="black", line_width=1)
    fig.update_layout(
        title='环比增长率时间序列',
        xaxis_title='日期',
        yaxis_title='增长率 (%)',
        height=500
    )
    fig.show()

In [ ]:
# 增长率分布
growth_cols = ['growth_mom', 'growth_qoq', 'growth_yoy']
available_growth_cols = [col for col in growth_cols if col in df.columns]

if available_growth_cols:
    fig, axes = plt.subplots(1, len(available_growth_cols), figsize=(15, 5))
    if len(available_growth_cols) == 1:
        axes = [axes]
    
    for i, col in enumerate(available_growth_cols):
        data = df[col].dropna() * 100
        axes[i].hist(data, bins=40, edgecolor='black', alpha=0.7)
        axes[i].axvline(0, color='red', linestyle='--', linewidth=2)
        axes[i].set_xlabel('增长率 (%)')
        axes[i].set_ylabel('频数')
        axes[i].set_title(f'{col.replace("growth_", "").upper()} 增长率分布')
        axes[i].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("增长率统计:")
    for col in available_growth_cols:
        data = df[col].dropna() * 100
        print(f"  {col}:")
        print(f"    平均值: {data.mean():.2f}%")
        print(f"    中位数: {data.median():.2f}%")
        print(f"    标准差: {data.std():.2f}%")
        print(f"    负增长占比: {(df[col] < 0).sum() / len(df) * 100:.1f}%")

## 5. 增长率分析

In [ ]:
# 季节性分析
if 'month' in df.columns:
    df_seasonal = df.groupby('month')['value_usd'].sum() / 1e6
    
    fig, ax = plt.subplots(figsize=(12, 6))
    df_seasonal.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_xlabel('月份')
    ax.set_ylabel('总贸易额 (百万美元)')
    ax.set_title('月度季节性模式')
    ax.set_xticklabels(range(1, 13), rotation=0)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    
    print("季节性分析:")
    print(f"  最高月份: {df_seasonal.idxmax()}月 (${df_seasonal.max():.2f}M)")
    print(f"  最低月份: {df_seasonal.idxmin()}月 (${df_seasonal.min():.2f}M)")
    print(f"  波动幅度: {(df_seasonal.max() - df_seasonal.min()) / df_seasonal.mean() * 100:.1f}%")

In [ ]:
# 移动平均趋势
ma_cols = ['value_ma_3m', 'value_ma_6m', 'value_ma_12m']
available_ma_cols = [col for col in ma_cols if col in df.columns]

if available_ma_cols:
    df_ma = df.groupby('date')[available_ma_cols].mean() / 1e6
    
    fig, ax = plt.subplots(figsize=(12, 6))
    df_ma.plot(ax=ax, linewidth=2)
    ax.set_xlabel('日期')
    ax.set_ylabel('贸易额 (百万美元)')
    ax.set_title('移动平均趋势分析')
    ax.legend(['3个月MA', '6个月MA', '12个月MA'])
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# 贸易价值时间序列
df_value = df.groupby(['date', 'hs_code'])['value_usd'].sum().reset_index()
df_value['value_million'] = df_value['value_usd'] / 1e6

fig = px.line(df_value, x='date', y='value_million', color='hs_code',
              title='贸易额时间序列 (按HS代码)',
              labels={'value_million': '贸易额 (百万美元)', 'date': '日期', 'hs_code': 'HS代码'})
fig.update_layout(height=500, hovermode='x unified')
fig.show()

print("贸易额统计:")
print(f"  总贸易额: ${df['value_usd'].sum() / 1e9:.2f}B")
print(f"  月均贸易额: ${df.groupby('date')['value_usd'].sum().mean() / 1e6:.2f}M")
print(f"  最大月度: ${df.groupby('date')['value_usd'].sum().max() / 1e6:.2f}M")
print(f"  最小月度: ${df.groupby('date')['value_usd'].sum().min() / 1e6:.2f}M")

## 4. 时间序列与趋势分析

In [ ]:
# 波动性时间序列
df_vol = df.groupby('date')['value_cov'].mean() * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_vol.index, y=df_vol.values, 
                         mode='lines+markers', name='平均CoV'))
fig.add_hline(y=30, line_dash="dash", line_color="red",
              annotation_text="高波动阈值 (30%)")
fig.update_layout(
    title='价值变异系数时间序列',
    xaxis_title='日期',
    yaxis_title='CoV (%)',
    height=500
)
fig.show()

In [ ]:
# 波动性指标分布
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 价值CoV分布
axes[0, 0].hist(df['value_cov'].dropna() * 100, bins=30, edgecolor='black', alpha=0.7, color='salmon')
axes[0, 0].axvline(30, color='red', linestyle='--', label='高波动 (>30%)')
axes[0, 0].set_xlabel('变异系数 (%)')
axes[0, 0].set_ylabel('频数')
axes[0, 0].set_title('价值变异系数 (CoV) 分布')
axes[0, 0].legend()

# 稳定性评分分布
axes[0, 1].hist(df['stability_score'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='lightgreen')
axes[0, 1].axvline(60, color='orange', linestyle='--', label='低稳定 (<60)')
axes[0, 1].set_xlabel('稳定性评分')
axes[0, 1].set_ylabel('频数')
axes[0, 1].set_title('稳定性评分分布 (0-100)')
axes[0, 1].legend()

# CoV vs 稳定性评分散点图
axes[1, 0].scatter(df['value_cov'] * 100, df['stability_score'], alpha=0.5, s=30)
axes[1, 0].set_xlabel('变异系数 (%)')
axes[1, 0].set_ylabel('稳定性评分')
axes[1, 0].set_title('波动性 vs 稳定性关系')
axes[1, 0].grid(True, alpha=0.3)

# 不同时间窗口的CoV比较
cov_cols = ['value_cov', 'value_cov_3m', 'value_cov_6m', 'value_cov_12m']
available_cov_cols = [col for col in cov_cols if col in df.columns]
if available_cov_cols:
    df[available_cov_cols].dropna().boxplot(ax=axes[1, 1])
    axes[1, 1].set_ylabel('变异系数')
    axes[1, 1].set_title('不同时间窗口的波动性比较')
    axes[1, 1].set_xticklabels(['全期', '3个月', '6个月', '12个月'], rotation=45)

plt.tight_layout()
plt.show()

print("波动性统计:")
print(f"  平均CoV: {df['value_cov'].mean() * 100:.2f}%")
print(f"  高波动 (>30%): {(df['value_cov'] > 0.3).sum()} ({(df['value_cov'] > 0.3).sum() / len(df) * 100:.1f}%)")
print(f"  平均稳定性评分: {df['stability_score'].mean():.2f}")
print(f"  低稳定 (<60): {(df['stability_score'] < 60).sum()} ({(df['stability_score'] < 60).sum() / len(df) * 100:.1f}%)")

## 3. 波动性与稳定性分析

In [ ]:
# Top N供应商集中度比较
top_cols = ['top1_share', 'top3_share', 'top5_share']
df_top = df[['date'] + top_cols].dropna()
df_top_mean = df_top.groupby('date')[top_cols].mean() * 100

fig, ax = plt.subplots(figsize=(12, 6))
df_top_mean.plot(ax=ax, marker='o')
ax.set_xlabel('日期')
ax.set_ylabel('市场份额 (%)')
ax.set_title('Top N供应商市场份额演变')
ax.legend(['Top 1供应商', 'Top 3供应商', 'Top 5供应商'])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("供应商集中度分析:")
for col in top_cols:
    print(f"  {col}: 平均 {df[col].mean() * 100:.1f}%, 最大 {df[col].max() * 100:.1f}%")

In [ ]:
# HHI时间序列 (使用Plotly交互式图表)
df_agg = df.groupby(['date', 'hs_code'])['hhi'].mean().reset_index()

fig = px.line(df_agg, x='date', y='hhi', color='hs_code',
              title='HHI时间序列趋势 (按HS代码)',
              labels={'hhi': 'HHI值', 'date': '日期', 'hs_code': 'HS代码'})

# 添加风险阈值线
fig.add_hline(y=1500, line_dash="dash", line_color="orange", 
              annotation_text="中等集中 (1500)")
fig.add_hline(y=2500, line_dash="dash", line_color="red",
              annotation_text="高度集中 (2500)")

fig.update_layout(height=500, hovermode='x unified')
fig.show()

In [ ]:
# HHI分布分析
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# HHI直方图
axes[0, 0].hist(df['hhi'].dropna(), bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(1500, color='orange', linestyle='--', label='中等集中 (1500)')
axes[0, 0].axvline(2500, color='red', linestyle='--', label='高度集中 (2500)')
axes[0, 0].set_xlabel('HHI值')
axes[0, 0].set_ylabel('频数')
axes[0, 0].set_title('HHI分布 (Herfindahl-Hirschman Index)')
axes[0, 0].legend()

# HHI箱线图（按HS代码）
df.boxplot(column='hhi', by='hs_code', ax=axes[0, 1])
axes[0, 1].set_xlabel('HS代码')
axes[0, 1].set_ylabel('HHI值')
axes[0, 1].set_title('各HS代码的HHI分布')
plt.sca(axes[0, 1])
plt.xticks(rotation=45)

# Top1供应商份额分布
axes[1, 0].hist(df['top1_share'].dropna() * 100, bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[1, 0].axvline(50, color='red', linestyle='--', label='高度依赖 (>50%)')
axes[1, 0].set_xlabel('Top1供应商份额 (%)')
axes[1, 0].set_ylabel('频数')
axes[1, 0].set_title('最大供应商市场份额分布')
axes[1, 0].legend()

# Gini系数分布
axes[1, 1].hist(df['gini_coefficient'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='lightblue')
axes[1, 1].axvline(0.4, color='orange', linestyle='--', label='中等不平等 (0.4)')
axes[1, 1].axvline(0.6, color='red', linestyle='--', label='高度不平等 (0.6)')
axes[1, 1].set_xlabel('Gini系数')
axes[1, 1].set_ylabel('频数')
axes[1, 1].set_title('Gini系数分布 (贸易不平等)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# 统计摘要
print(f"HHI统计:")
print(f"  平均值: {df['hhi'].mean():.2f}")
print(f"  中位数: {df['hhi'].median():.2f}")
print(f"  高度集中 (>2500): {(df['hhi'] > 2500).sum()} ({(df['hhi'] > 2500).sum() / len(df) * 100:.1f}%)")
print(f"\nTop1供应商份额:")
print(f"  平均值: {df['top1_share'].mean() * 100:.2f}%")
print(f"  高度依赖 (>50%): {(df['top1_share'] > 0.5).sum()} ({(df['top1_share'] > 0.5).sum() / len(df) * 100:.1f}%)")

## 2. 供应链集中度分析

In [ ]:
# 基础统计信息
feature_cols = [c for c in df.columns if c not in ['date', 'hs_code', 'hs_description', 'country', 'value_usd', 'quantity', 'unit', 'trade_type', 'data_source']]
print(f"特征统计 (共{len(feature_cols)}个特征):\n")
df[feature_cols].describe().T

In [ ]:
# 缺失值分析
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    '缺失数量': missing,
    '缺失百分比': missing_pct
})
missing_df = missing_df[missing_df['缺失数量'] > 0].sort_values('缺失数量', ascending=False)

if len(missing_df) > 0:
    print("缺失值统计:")
    print(missing_df)
else:
    print("✅ 无缺失值")

In [ ]:
# 数据类型和列信息
print("数据列信息:")
print("\n原始数据列 (9):")
original_cols = ['date', 'hs_code', 'hs_description', 'country', 'value_usd', 'quantity', 'unit', 'trade_type', 'data_source']
for col in original_cols:
    if col in df.columns:
        print(f"  - {col}: {df[col].dtype}")

print(f"\n特征列 ({len([c for c in df.columns if c not in original_cols])}):")
feature_cols = [c for c in df.columns if c not in original_cols]
print(f"  集中度特征: {len([c for c in feature_cols if 'hhi' in c or 'top' in c or 'gini' in c or 'supplier' in c])}")
print(f"  波动性特征: {len([c for c in feature_cols if 'std' in c or 'cov' in c or 'volatility' in c or 'stability' in c])}")
print(f"  时间序列特征: {len([c for c in feature_cols if 'trend' in c or 'ma_' in c or 'lag_' in c or 'momentum' in c or 'season' in c or 'month' in c or 'quarter' in c or 'year' in c])}")
print(f"  增长特征: {len([c for c in feature_cols if 'growth' in c or 'cagr' in c or 'cumulative' in c or 'growing' in c])}")

In [ ]:
# 加载特征数据
df = pd.read_parquet('../data/processed/features_sample.parquet')

print(f"数据形状: {df.shape}")
print(f"时间范围: {df['date'].min()} 到 {df['date'].max()}")
print(f"HS代码数量: {df['hs_code'].nunique()}")
print(f"国家数量: {df['country'].nunique()}")
print(f"\n前5行数据:")
df.head()

## 1. 数据加载与概览

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

# 设置
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

print("✅ 库导入完成")

# 供应链风险特征分析 (Supply Chain Risk Feature Analysis)

**项目**: SCRAM (Supply Chain Risk Analysis Model)  
**数据**: 半导体集成电路进口数据 (2010-2019)  
**特征**: 49个风险特征 (集中度、波动性、时间序列、增长率)

---

## 分析目标

1. 理解特征分布和统计特性
2. 识别供应链集中度和波动性模式
3. 分析时间序列趋势和季节性
4. 发现高风险时期和异常模式
5. 为模型训练提供特征选择建议